# 20 - V2 robustness evaluation

Run paired deterministic feature masking and bounded Gaussian perturbation tests.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/gdrive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/gdrive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').is_dir():
        from google.colab import drive
        for mountpoint in (Path('/content/drive'), Path('/content/gdrive')):
            if mountpoint.is_dir() and any(mountpoint.iterdir()):
                continue
            try:
                drive.mount(str(mountpoint), force_remount=False)
            except ValueError as exc:
                if 'must not already contain files' not in str(exc):
                    raise
                continue
            for candidate in DRIVE_CANDIDATES:
                if is_project_root(candidate):
                    return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError('Canonical RareGuard-XIDS IMPLEMENTATION project not found.')
os.chdir(PROJECT_ROOT)
PROJECT_SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, PROJECT_SRC)
os.environ['PYTHONPATH'] = PROJECT_SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project root:', PROJECT_ROOT)
print('Runtime:', 'Colab' if Path('/content').is_dir() else 'local')
print('Python path ready')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Runtime: Colab
Python path ready


In [2]:
import subprocess
import torch

DEVELOPMENT_DATASETS = [
    'cse_cic_ids2018',
    'edge_iiotset',
    'cic_bccc_nrc_tabulariotattack_2024',
    'cic_iiot_2025_datasense',
]
CONFIRMATION_DATASETS = ['ciciot2023', 'ton_iot']

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for the development-dataset stage.')

def run_device_split_stage(module: str) -> None:
    for dataset in DEVELOPMENT_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cuda', '--resume'],
            check=True,
        )
    for dataset in CONFIRMATION_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cpu', '--resume'],
            check=True,
        )

run_device_split_stage('rareguard.experiments.run_publication_robustness')


In [3]:
import pandas as pd
display(pd.read_csv('artifacts/publication_v2/results/robustness_summary.csv'))


,dataset,model,perturbation,strength,seeds,f1_macro_mean,f1_macro_std,rare_f1_macro_mean,rare_f1_macro_std
0,cic_bccc_nrc_tabulariotattack_2024,NOVA-XIDS,feature_mask,0.10,5,0.490244,0.028274,0.044472,0.009337
1,cic_bccc_nrc_tabulariotattack_2024,NOVA-XIDS,feature_mask,0.25,5,0.406469,0.009081,0.022092,0.008921
2,cic_bccc_nrc_tabulariotattack_2024,NOVA-XIDS,gaussian_noise,0.05,5,0.480232,0.011330,0.106440,0.019134
3,cic_bccc_nrc_tabulariotattack_2024,NOVA-XIDS,gaussian_noise,0.10,5,0.408780,0.021276,0.077783,0.023407
4,cic_bccc_nrc_tabulariotattack_2024,RareGuard-XIDS,feature_mask,0.10,5,0.530721,0.012630,0.080229,0.056466
5,cic_bccc_nrc_tabulariotattack_2024,RareGuard-XIDS,feature_mask,0.25,5,0.448639,0.018416,0.047350,0.028770
6,cic_bccc_nrc_tabulariotattack_2024,RareGuard-XIDS,gaussian_noise,0.05,5,0.511409,0.003987,0.124450,0.037385
7,cic_bccc_nrc_tabulariotattack_2024,RareGuard-XIDS,gaussian_noise,0.10,5,0.461505,0.013392,0.116096,0.043768
8,cic_iiot_2025_datasense,NOVA-XIDS,feature_mask,0.10,5,0.297858,0.027351,0.323970,0.044707
9,cic_iiot_2025_datasense,NOVA-XIDS,feature_mask,0.25,5,0.247293,0.021402,0.260084,0.031718
